# Sample Size Calculation Using Simulations

## Overview

This notebook demonstrates how to calculate sample size for a planned test of significance using simulations. I'll build on your existing knowledge of simulating null distributions and computing p-values, extending it to power analysis and sample size determination.

## Learning Objectives

By the end of this notebook, you will be able to use simulations to choose a sample size.

## The 5-Step Process for Sample Size Calculation

1. **State the maximum acceptable probability of a Type I error**: This becomes the level of significance (α)
2. **State the minimum acceptable power**: (1 - power = Probability of Type II error)
3. **Estimate the minimum useful value for the alternative hypothesis**
4. **Estimate other quantities as needed** using previous studies or your intuition
5. **Compute the sample size** based on 1-4 using simulations


In [48]:
# Import necessary libraries
import numpy as np
#import matplotlib.pyplot as plt
from scipy import stats

# Set random seed for reproducibility
np.random.seed(42)

## Step 1: Understanding Type I and Type II Errors

Before we can calculate sample size, we need to understand the two types of errors in hypothesis testing:

### Type I Error (α)
- **Definition**: Rejecting the null hypothesis when it's actually true
- **Consequence**: False discovery - claiming an effect exists when it doesn't
- **Probability**: α (alpha) - this is our significance level: we reject the null hypothesis if the p-value is less than this quantity.  It is also an upper bound for the probability of a Type I error!  This is because a p-value is the probability of seeing new data like the study assuming the null is true.
- **Typical values**: 0.05 (5%), 0.01 (1%), 0.10 (10%)

### Type II Error (β)
- **Definition**: Failing to reject the null hypothesis when the alternative is actually true
- **Consequence**: Missed Opportunity - missing a real effect
- **Probability**: β (beta)

### Power (1 - β)
- **Definition**: The probability of correctly rejecting the null hypothesis when the alternative is true
- **Interpretation**: The probability of detecting a real effect
- **Typical values**: 0.80 (80%), 0.90 (90%), 0.95 (95%)


In an ideal world, we would minimize both Type I and Type II errors. In practice, we first fix the maximum acceptable probability of a Type I error (α, the significance level), because false positives are often considered more serious. Then, for a given effect size, we choose the sample size (n) so that the test has enough power (1 - β) to reliably detect a real effect. This ensures our study is both trustworthy (low chance of false discovery) and sensitive (high chance of detecting a true effect).

## Step 2: Setting Up Our Example

Let's work through a concrete example: **comparing the mean proportion of landfill waste that is actually landfill from audits before and after recommendations were made**.

### Research Question
Let $p_1$ be the fraction of total waste that is actually landfill before recommendations were made.
Let $p_2$ be the fraction of total waste that is actually landfill after recommendations were made.

### Hypotheses
- **H₀**: $p_1 = p_2$ (no difference in proportions of total waste that is actually landfill)
- **H₁**: $p_1 < p_2$ (increase in proportion of total waste that is landfill after the recommendations)

### Test
Permutation Test:

Use the difference in proportions (after - before) as the test statistic:
$$ t = \frac{\sum_i L_{2i}}{\sum_i T_{2i}} - \frac{\sum_i L_{1i}}{\sum_i T_{1i}} $$

where $i$ is the index over audits, $L_{1i}$ is the weight of landfill $i$ th audit before the recommendation and $T_{1i}$ is the total weight of the $i$ th audit before the recommendation.

Steps in the Permutation Test:

a. Compute the actual test statistic using the data from audits

b. Permute the labels of before and after then compute the test statistic

c. Repeat Step b many times

d. Estimate the p-value by computing the proportion of times the permuted test statistics were at least as large as the actual test statistic.

e. Reject the null hypothesis in favor of the alternative if the p-value is less than the level of significance.




In [49]:
# Step 1: Set our significance level and desired power
alpha = 0.05  # 5% chance of Type I error
power = 0.80  # 80% chance of detecting a real effect

print(f"Significance level (α) = {alpha}")
print(f"Desired power = {power}")
print(f"Type II error rate (β) = {1 - power}")


Significance level (α) = 0.05
Desired power = 0.8
Type II error rate (β) = 0.19999999999999996


## Step 3: Defining Our Effect Size

We need to specify what constitutes a meaningful difference. This is often the most challenging part of sample size calculation.

### Effect Size
For our tutoring example, let's say:
- **Proportion of waste that is landfill before**: $p_1 = 0.5$
- **Proportion of waste that is landfill after**: $p_2 = 0.75$
- **Effect size**: 0.25 difference




In [78]:
# Step 2: Define our parameters

p1 = 0.5
p2 = 0.8

# Set random seed for reproducibility
np.random.seed(42)

# Define population sizes (total weight in pounds)
total_weight1 = 1000
total_weight2 = 1000

# Assume each piece of trash has a random weight between 0.1 and 5 pounds
# We'll generate pieces until the total weight is reached for each population

def generate_trash_population(total_weight, landfill_proportion):
    pieces = []
    landfill_labels = []
    current_weight = 0
    while current_weight < total_weight:
        # Random weight for this piece
        piece_weight = np.random.uniform(0.1, 5.0)
        # If adding this piece exceeds the total, trim it
        if current_weight + piece_weight > total_weight:
            piece_weight = total_weight - current_weight
        pieces.append(piece_weight)
        # Assign landfill label with probability = landfill_proportion
        is_landfill = np.random.rand() < landfill_proportion
        landfill_labels.append(is_landfill)
        current_weight += piece_weight
    return np.array(pieces), np.array(landfill_labels)

# Generate both populations
pop1_weights, pop1_landfill = generate_trash_population(total_weight1, p1)
pop2_weights, pop2_landfill = generate_trash_population(total_weight2, p2)




## Step 4: Simulating a Single Test

Let's start by simulating what happens when we run our test with a specific sample size. This will help us understand the concept before we search for the optimal sample size.

### The Simulation Process
1. Generate data for both groups under the alternative hypothesis (with the effect)
2. Perform the Permutation as described in steps a-e.
3. Check if we reject the null hypothesis (p-value < α)
4. Repeat this many times to estimate power


In [79]:
# Step 3: Simulate a single test with sample size n before and n after
n = 35

# Generate data under the alternative hypothesis (with effect)

# Randomly select n indices from each population (without replacement)
group1_indices = np.random.choice(len(pop1_weights), size=n, replace=False)
group2_indices = np.random.choice(len(pop2_weights), size=n, replace=False)

# Use these indices to select the corresponding weights and landfill labels
group1_data = pop1_weights[group1_indices]
group1_landfill = pop1_landfill[group1_indices]

group2_data = pop2_weights[group2_indices]
group2_landfill = pop2_landfill[group2_indices]

# Calculate the actual test statistic:

# Compute the proportion of landfill by weight for each group then difference
group1_landfill_weight = group1_data[group1_landfill].sum()
group2_landfill_weight = group2_data[group2_landfill].sum()

group1_total_weight = group1_data.sum()
group2_total_weight = group2_data.sum()

group1_landfill_prop = group1_landfill_weight / group1_total_weight if group1_total_weight > 0 else np.nan
group2_landfill_prop = group2_landfill_weight / group2_total_weight if group2_total_weight > 0 else np.nan

actual_prop_diff = group1_landfill_prop - group2_landfill_prop

# CHANGED so that the labels and weights are shuffled together
# Permutation test: shuffle (weight, label) PAIRS, recompute statistic, repeat many times
n_permutations = 10000
n1 = len(group1_data)
pairs = list(zip(np.concatenate([group1_data, group2_data]),
                 np.concatenate([group1_landfill, group2_landfill])))
perm_stats = []
for _ in range(n_permutations):
    np.random.shuffle(pairs)  # shuffles in place
    # split back into two groups of size n1 and n2
    g1_pairs = pairs[:n1]
    g2_pairs = pairs[n1:]

    g1_weights = np.array([w for w, _ in g1_pairs])
    g1_labels  = np.array([l for _, l in g1_pairs], dtype=bool)
    g2_weights = np.array([w for w, _ in g2_pairs])
    g2_labels  = np.array([l for _, l in g2_pairs], dtype=bool)

    # weighted landfill proportions for the permuted split
    g1_landfill_weight = g1_weights[g1_labels].sum()
    g2_landfill_weight = g2_weights[g2_labels].sum()
    g1_total = g1_weights.sum()
    g2_total = g2_weights.sum()

    g1_prop = g1_landfill_weight / g1_total if g1_total > 0 else np.nan
    g2_prop = g2_landfill_weight / g2_total if g2_total > 0 else np.nan

    perm_stats.append(g1_prop - g2_prop)

perm_stats = np.array(perm_stats)

# Two-sided p-value
p_value = np.mean(np.abs(perm_stats) >= np.abs(actual_prop_diff))
print(f"Permutation test p-value: {p_value:.4f}")


Permutation test p-value: 0.0290


## Step 5: Estimating Power Through Simulation

Now let's run this simulation many times to estimate the power. Power is the proportion of times we correctly reject the null hypothesis when the alternative is true.

### Power Calculation
- Run the test many times (e.g., 1000 simulations)
- Count how many times we reject H₀ (p < α)
- Power = (Number of rejections) / (Total simulations)


In [81]:
# Step 4: Estimate power with n = n  # Use n from the previous cell
n = 25
n_simulations = 1000
alpha = 0.
rejections = 0

# Build paired populations (weight, label) for BEFORE and AFTER
pop1_pairs = list(zip(pop1_weights, pop1_landfill))
pop2_pairs = list(zip(pop2_weights, pop2_landfill))

for sim in range(n_simulations):
    # --- draw a fresh experiment under the alternative: sample pairs (not weights/labels separately) ---
    # (choose with or without replacement; without replacement is typical for finite-population sampling)
    g1_idx = np.random.choice(len(pop1_pairs), size=n, replace=False)
    g2_idx = np.random.choice(len(pop2_pairs), size=n, replace=False)

    g1_pairs = [pop1_pairs[i] for i in g1_idx]
    g2_pairs = [pop2_pairs[i] for i in g2_idx]

    g1_w = np.array([w for w, _ in g1_pairs])
    g1_L = np.array([L for _, L in g1_pairs], dtype=bool)
    g2_w = np.array([w for w, _ in g2_pairs])
    g2_L = np.array([L for _, L in g2_pairs], dtype=bool)

    # observed statistic for this simulated experiment
    g1_prop = (g1_w[g1_L].sum() / g1_w.sum()) if g1_w.sum() > 0 else np.nan
    g2_prop = (g2_w[g2_L].sum() / g2_w.sum()) if g2_w.sum() > 0 else np.nan
    T_obs = g1_prop - g2_prop

    # --- permutation test inside this simulation: permute WHOLE PAIRS across groups ---
    pairs_all = g1_pairs + g2_pairs
    n1 = len(g1_pairs)
    B = 1000  # permutations per simulation
    perm_stats = []

    for _ in range(B):
        np.random.shuffle(pairs_all)  # shuffles in place
        g1p = pairs_all[:n1]
        g2p = pairs_all[n1:]

        g1w = np.array([w for w, _ in g1p])
        g1L = np.array([L for _, L in g1p], dtype=bool)
        g2w = np.array([w for w, _ in g2p])
        g2L = np.array([L for _, L in g2p], dtype=bool)

        g1p_prop = (g1w[g1L].sum() / g1w.sum()) if g1w.sum() > 0 else np.nan
        g2p_prop = (g2w[g2L].sum() / g2w.sum()) if g2w.sum() > 0 else np.nan
        perm_stats.append(g1p_prop - g2p_prop)

    perm_stats = np.array(perm_stats)
    p_val = np.mean(np.abs(perm_stats) >= np.abs(T_obs))
    if p_val < alpha:
        rejections += 1

power_estimate = rejections / n_simulations
print(f"Rejections: {rejections} out of {n_simulations}")
print(f"Estimated power: {power_estimate:.3f}")


Rejections: 820 out of 1000
Estimated power: 0.820


## Step 6: Finding the Required Sample Size

The power with n = 3 is likely too low. We need to find the sample size that gives us our target power of 0.80.

### Sample Size Search Strategy
1. Start with a reasonable guess
2. Test the power at that sample size
3. If power is too low, increase sample size
4. If power is too high, decrease sample size
5. Repeat until we find the right sample size


## Key Takeaways

You can play around with the choices of alpha, beta, n and the "true" altnerative hypothesis to see that:

1. **Power increases with sample size**: Larger samples give us more power to detect effects
2. **There's a trade-off**: More power requires more participants (and more cost/time)
3. **Effect size matters**: Larger effects are easier to detect with smaller samples
4. **Significance level matters**: Stricter significance levels (smaller α) require larger samples


### What to Think About for Your Own Project
- **Sample size planning is crucial**: Without proper planning, you might miss real effects (low power) or waste resources (overpowered studies)
- **Simulation is flexible**: This approach works for any statistical test
- **Real-world constraints**: Consider practical limitations (time, money, participant availability) when setting your target power

To use this approach for your own research:
1. **Define your research question** and statistical test
2. **Set your significance level**, carefully considering the impact of a Type I error
3. **Choose your target power**, carefully considering the impact of a Type II error (Power = 1-P(type II))
4. **Estimate your effect size** based on previous research or pilot studies
5. **Write the simulation of your data generating process** with your chosen statistical test applied to find your required sample size
6. **Consider practical constraints** and adjust if necessary

### Common Mistakes to Avoid

- **Underestimating effect size**: If you overestimate the effect, you'll need fewer participants but might miss smaller, still meaningful effects
- **Ignoring practical constraints**: Don't plan for sample sizes you can't realistically achieve
- **Forgetting about attrition**: Plan for some participants to drop out
- **Not considering multiple comparisons**: If you're running multiple tests, you might need to adjust your significance level
